<a href="https://colab.research.google.com/github/sangharshmandavkar/python_tutorial-/blob/main/app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q streamlit pandas requests dnspython python-whois pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 9.0 MB/s eta 0:00:00


In [9]:
%%writefile parser.py

from email import policy
from email.parser import BytesParser
from email.utils import parseaddr
import re


def parse_eml_bytes(data: bytes):
    msg = BytesParser(policy=policy.default).parsebytes(data)

    subject = msg.get("Subject", "")
    from_header = msg.get("From", "")
    reply_to = msg.get("Reply-To", "")
    return_path = msg.get("Return-Path", "")

    sender_name, sender_email = parseaddr(from_header)
    _, reply_email = parseaddr(reply_to)
    _, return_email = parseaddr(return_path)

    body = ""

    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()

            if content_type == "text/plain":
                try:
                    body += part.get_content()
                except Exception:
                    try:
                        body += part.get_payload(decode=True).decode(
                            "utf-8", errors="ignore"
                        )
                    except Exception:
                        pass
    else:
        try:
            body = msg.get_content()
        except Exception:
            payload = msg.get_payload(decode=True)
            if payload:
                body = payload.decode("utf-8", errors="ignore")

    headers = dict(msg.items())

    received_headers = msg.get_all("Received", [])

    return {
        "subject": subject,
        "from": from_header,
        "sender_name": sender_name,
        "sender_email": sender_email,
        "reply_to": reply_email,
        "return_path": return_email,
        "body": body,
        "headers": headers,
        "received": received_headers,
        "message_id": msg.get("Message-ID", ""),
        "date": msg.get("Date", ""),
        "to": msg.get("To", ""),
        "content_type": msg.get_content_type(),
    }

Writing parser.py


In [10]:
%%writefile risk_engine.py

import re
from dataclasses import dataclass


@dataclass
class RiskThresholds:
    high: int = 70
    medium: int = 40


def score_email(parsed, thresholds=None, do_geo_lookup=False):
    if thresholds is None:
        thresholds = RiskThresholds()

    subject = str(parsed.get("subject", ""))
    body = str(parsed.get("body", ""))
    sender = str(parsed.get("sender_email", ""))
    reply_to = str(parsed.get("reply_to", ""))

    text = f"{subject} {body}".lower()

    score = 0
    reasons = []

    # Suspicious keywords
    keywords = [
        "urgent",
        "verify your account",
        "verify account",
        "password",
        "reset password",
        "login",
        "click here",
        "suspended",
        "bank",
        "payment",
        "invoice",
        "wire transfer",
        "gift card",
        "otp",
        "credential",
        "security alert",
    ]

    found_keywords = []

    for keyword in keywords:
        if keyword in text:
            found_keywords.append(keyword)

    if found_keywords:
        score += min(40, len(found_keywords) * 5)
        reasons.append(
            "Suspicious keywords detected: "
            + ", ".join(found_keywords[:8])
        )

    # URL detection
    urls = re.findall(
        r"https?://[^\s<>\"']+",
        body,
        flags=re.IGNORECASE
    )

    if urls:
        score += min(25, len(urls) * 5)
        reasons.append(f"{len(urls)} URL(s) detected in email body")

    # IP-based URL
    ip_urls = re.findall(
        r"https?://(?:\d{1,3}\.){3}\d{1,3}",
        body,
        flags=re.IGNORECASE
    )

    if ip_urls:
        score += 20
        reasons.append("URL contains an IP address")

    # Reply-To mismatch
    if reply_to and sender:
        sender_domain = sender.split("@")[-1].lower()
        reply_domain = reply_to.split("@")[-1].lower()

        if sender_domain != reply_domain:
            score += 20
            reasons.append("Reply-To domain differs from sender domain")

    # Suspicious sender patterns
    suspicious_sender_words = [
        "admin",
        "support",
        "security",
        "billing",
        "payment",
        "verify"
    ]

    if any(word in sender.lower() for word in suspicious_sender_words):
        score += 5
        reasons.append("Sender address contains a commonly impersonated role")

    # Determine risk tier
    score = min(score, 100)

    if score >= thresholds.high:
        tier = "HIGH"
    elif score >= thresholds.medium:
        tier = "MEDIUM"
    else:
        tier = "LOW"

    # Basic fraud classification
    fraud_detection = {
        "is_fraud": score >= thresholds.medium,
        "confidence": score,
        "reasons": reasons
    }

    # Basic authentication information
    headers = parsed.get("headers", {})

    auth_results = headers.get("Authentication-Results", "")

    spf = "Unknown"
    dkim = "Unknown"
    dmarc = "Unknown"

    if "spf=pass" in auth_results.lower():
        spf = "Pass"
    elif "spf=fail" in auth_results.lower():
        spf = "Fail"

    if "dkim=pass" in auth_results.lower():
        dkim = "Pass"
    elif "dkim=fail" in auth_results.lower():
        dkim = "Fail"

    if "dmarc=pass" in auth_results.lower():
        dmarc = "Pass"
    elif "dmarc=fail" in auth_results.lower():
        dmarc = "Fail"

    return {
        "risk_score": score,
        "tier": tier,
        "reasons": reasons,
        "fraud_detection": fraud_detection,
        "geo_intel": {
            "enabled": bool(do_geo_lookup),
            "status": "Not implemented in Colab-safe version"
        },
        "authentication": {
            "spf": spf,
            "dkim": dkim,
            "dmarc": dmarc
        },
        "urls": urls,
        "sender": sender,
        "subject": subject
    }

Writing risk_engine.py


In [11]:
%%writefile blockchain_log.py

import hashlib
import json
import os
from datetime import datetime


CHAIN_FILE = "email_evidence_chain.json"


def _hash_block(block):
    block_copy = dict(block)
    block_copy.pop("hash", None)

    raw = json.dumps(
        block_copy,
        sort_keys=True,
        default=str
    ).encode("utf-8")

    return hashlib.sha256(raw).hexdigest()


def _load_chain():
    if not os.path.exists(CHAIN_FILE):
        return []

    try:
        with open(CHAIN_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return []


def _save_chain(chain):
    with open(CHAIN_FILE, "w", encoding="utf-8") as f:
        json.dump(chain, f, indent=2, default=str)


def add_evidence(record):
    chain = _load_chain()

    previous_hash = (
        chain[-1]["hash"]
        if chain
        else "GENESIS"
    )

    block = {
        "index": len(chain),
        "timestamp": datetime.utcnow().isoformat(),
        "previous_hash": previous_hash,
        "record": record
    }

    block["hash"] = _hash_block(block)

    chain.append(block)
    _save_chain(chain)

    return block


def verify_chain():
    chain = _load_chain()

    if not chain:
        return True

    for i, block in enumerate(chain):

        # Verify current block hash
        expected_hash = _hash_block(block)

        if block.get("hash") != expected_hash:
            return False

        # Verify previous block connection
        if i == 0:
            if block.get("previous_hash") != "GENESIS":
                return False
        else:
            if block.get("previous_hash") != chain[i - 1].get("hash"):
                return False

    return True


def get_chain():
    return _load_chain()

Writing blockchain_log.py


In [12]:
%%writefile app.py

import streamlit as st
import pandas as pd

from parser import parse_eml_bytes
from risk_engine import score_email, RiskThresholds
from blockchain_log import verify_chain, get_chain, add_evidence


st.set_page_config(
    page_title="AI Email Threat Detection",
    page_icon="🛡️",
    layout="wide"
)


st.title("🛡️ AI-Powered Email Threat Detection")
st.caption(
    "Phishing, fraud and suspicious email analysis with forensic evidence logging"
)


# Sidebar
st.sidebar.header("Risk Configuration")

high_threshold = st.sidebar.slider(
    "High Risk Threshold",
    min_value=50,
    max_value=100,
    value=70
)

medium_threshold = st.sidebar.slider(
    "Medium Risk Threshold",
    min_value=20,
    max_value=80,
    value=40
)

do_geo = st.sidebar.checkbox(
    "Enable Geo Intelligence",
    value=False
)


# Blockchain status
st.sidebar.header("Evidence Chain")

try:
    chain_valid = verify_chain()

    if chain_valid:
        st.sidebar.success("Evidence chain valid")
    else:
        st.sidebar.error("Evidence chain compromised")

except Exception as e:
    st.sidebar.warning(f"Chain check failed: {e}")


# Upload email
uploaded_file = st.file_uploader(
    "Upload a raw email (.eml)",
    type=["eml"]
)


if uploaded_file is None:

    st.info(
        "Upload an .eml email file to start threat analysis."
    )

    st.markdown("""
### Features

- 🔍 Email header analysis
- 🎣 Phishing and fraud detection
- 🛡️ SPF / DKIM / DMARC inspection
- 🔗 URL detection
- 👤 Sender analysis
- 📊 Risk scoring
- ⛓️ Tamper-evident evidence chain
""")

else:

    try:

        # Read email
        email_bytes = uploaded_file.getvalue()

        parsed = parse_eml_bytes(email_bytes)

        # Risk thresholds
        thresholds = RiskThresholds(
            high=high_threshold,
            medium=medium_threshold
        )

        # Analyze
        report = score_email(
            parsed,
            thresholds=thresholds,
            do_geo_lookup=do_geo
        )

        # Extract results safely
        risk_score = report.get("risk_score", 0)
        tier = report.get("tier", "UNKNOWN")

        sender = parsed.get(
            "sender_email",
            "Unknown"
        )

        subject = parsed.get(
            "subject",
            "No Subject"
        )

        body = parsed.get(
            "body",
            ""
        )

        # Header
        st.subheader("📊 Threat Analysis")

        col1, col2, col3 = st.columns(3)

        with col1:
            st.metric(
                "Risk Score",
                f"{risk_score}/100"
            )

        with col2:
            st.metric(
                "Risk Level",
                tier
            )

        with col3:
            st.metric(
                "Email Size",
                f"{len(email_bytes)} bytes"
            )


        # Risk message
        if tier == "HIGH":
            st.error(
                "🚨 HIGH RISK: This email contains multiple suspicious indicators."
            )

        elif tier == "MEDIUM":
            st.warning(
                "⚠️ MEDIUM RISK: This email contains suspicious indicators."
            )

        else:
            st.success(
                "✅ LOW RISK: No major suspicious indicators detected."
            )


        # Tabs
        tab1, tab2, tab3, tab4 = st.tabs([
            "📧 Email Details",
            "🔐 Authentication",
            "🔎 Forensics",
            "⛓️ Evidence Chain"
        ])


        # TAB 1
        with tab1:

            st.subheader("Email Information")

            details = {
                "From": parsed.get("from", ""),
                "Sender Email": sender,
                "Reply-To": parsed.get("reply_to", ""),
                "Return-Path": parsed.get("return_path", ""),
                "To": parsed.get("to", ""),
                "Subject": subject,
                "Date": parsed.get("date", ""),
                "Message-ID": parsed.get("message_id", "")
            }

            details_df = pd.DataFrame(
                list(details.items()),
                columns=["Field", "Value"]
            )

            st.dataframe(
                details_df,
                use_container_width=True,
                hide_index=True
            )


            st.subheader("Email Body")

            if body.strip():
                st.text_area(
                    "Extracted Body",
                    body,
                    height=300
                )
            else:
                st.info("No readable text body found.")


        # TAB 2
        with tab2:

            st.subheader("SPF / DKIM / DMARC")

            auth = report.get(
                "authentication",
                {}
            )

            auth_df = pd.DataFrame({
                "Protocol": [
                    "SPF",
                    "DKIM",
                    "DMARC"
                ],
                "Result": [
                    auth.get("spf", "Unknown"),
                    auth.get("dkim", "Unknown"),
                    auth.get("dmarc", "Unknown")
                ]
            })

            st.dataframe(
                auth_df,
                use_container_width=True,
                hide_index=True
            )


            st.subheader("Authentication-Results Header")

            headers = parsed.get(
                "headers",
                {}
            )

            st.code(
                headers.get(
                    "Authentication-Results",
                    "Not available"
                )
            )


        # TAB 3
        with tab3:

            st.subheader("Forensic Indicators")

            reasons = report.get(
                "reasons",
                []
            )

            if reasons:

                for reason in reasons:
                    st.warning(f"⚠️ {reason}")

            else:

                st.success(
                    "No major suspicious indicators detected."
                )


            st.subheader("Detected URLs")

            urls = report.get(
                "urls",
                []
            )

            if urls:

                for url in urls:
                    st.code(url)

            else:

                st.info("No URLs detected.")


            st.subheader("Raw Headers")

            headers = parsed.get(
                "headers",
                {}
            )

            header_rows = []

            for key, value in headers.items():
                header_rows.append({
                    "Header": key,
                    "Value": value
                })

            if header_rows:

                header_df = pd.DataFrame(header_rows)

                st.dataframe(
                    header_df,
                    use_container_width=True,
                    hide_index=True
                )


        # TAB 4
        with tab4:

            st.subheader(
                "Tamper-Evident Evidence Chain"
            )

            if st.button("Log Analysis to Evidence Chain"):

                evidence = {
                    "filename": uploaded_file.name,
                    "sender": sender,
                    "subject": subject,
                    "risk_score": risk_score,
                    "risk_level": tier
                }

                try:

                    block = add_evidence(evidence)

                    st.success(
                        f"Evidence logged successfully. Block #{block['index']}"
                    )

                except Exception as e:

                    st.error(
                        f"Unable to log evidence: {e}"
                    )


            try:

                chain = get_chain()

                if chain:

                    chain_rows = []

                    for block in chain:

                        chain_rows.append({
                            "Index": block.get("index"),
                            "Timestamp": block.get("timestamp"),
                            "Risk": block.get("record", {}).get(
                                "risk_level",
                                ""
                            ),
                            "Hash": block.get("hash", "")[:20] + "..."
                        })

                    st.dataframe(
                        pd.DataFrame(chain_rows),
                        use_container_width=True,
                        hide_index=True
                    )

                else:

                    st.info(
                        "No evidence has been logged yet."
                    )

            except Exception as e:

                st.error(
                    f"Unable to read evidence chain: {e}"
                )


    except Exception as e:

        st.error("❌ Error while processing the email")

        st.exception(e)

Writing app.py


In [13]:
%%writefile sample_phishing.eml

From: Security Team <security@example.com>
To: user@example.com
Subject: URGENT - Verify your account immediately
Reply-To: attacker@fake-example.com
Date: Wed, 24 Sep 2026 10:00:00 +0530
Message-ID: <test123@example.com>
Authentication-Results: spf=fail; dkim=fail; dmarc=fail
Content-Type: text/plain

URGENT!

Your account has been suspended.

Please verify your account immediately by clicking here:

http://192.168.1.100/verify

Enter your password and OTP to restore access.

Writing sample_phishing.eml


In [14]:
import os

print("Files created:")

for file in [
    "app.py",
    "parser.py",
    "risk_engine.py",
    "blockchain_log.py",
    "sample_phishing.eml"
]:
    print(file, "✅" if os.path.exists(file) else "❌")

Files created:
app.py ✅
parser.py ✅
risk_engine.py ✅
blockchain_log.py ✅
sample_phishing.eml ✅


In [16]:
!streamlit run app.py &>/content/streamlit.log &

In [21]:
from pyngrok import ngrok

ngrok.set_auth_token("3JuFaI57HCGtD57yRtL6NjYHgxd_4YmwPJQC3CsUeUYXhqKPS")

public_url = ngrok.connect(8501)

print("Streamlit URL:")
print(public_url)

Streamlit URL:
NgrokTunnel: "https://decency-wildfowl-sponge.ngrok-free.dev" -> "http://localhost:8501"
